# Genomics Analysis Agent - Cancer Mutation Analysis

This notebook demonstrates how to use the **GenomicsAnalysisAgent** for analyzing somatic mutations in cancer genomics research.

The agent can perform:
- Driver gene identification
- Mutation type distribution analysis
- Pathway enrichment analysis
- Tumor mutational burden (TMB) calculation
- Actionable mutation discovery
- Mutation visualization (oncoprints, frequency plots)

## Use Cases
- Cancer driver gene discovery
- Pathway-level mutation analysis
- Precision medicine target identification
- Clinical trial patient stratification
- Biomarker discovery

## Setup

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import os

from langchain_openai import ChatOpenAI
from ai_data_science_team.ml_agents import GenomicsAnalysisAgent

In [ ]:
# Set your OpenAI API key
os.environ['OPENAI_API_KEY'] = "your-api-key-here"

# Initialize language model
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
llm

## Cost-Effective Alternative: OpenRouter

**Save 10-100x on API costs** with OpenRouter!

Cancer genomics analysis can generate large outputs (gene lists, pathway reports, mutation tables), leading to high token usage. OpenRouter helps reduce costs dramatically:

- **Claude 3.5 Sonnet**: ~$3/M tokens (recommended for genomics)
- **Claude 3 Haiku**: ~$0.25/M tokens (fast & cheap)  
- **Gemini Pro 1.5**: ~$1.25/M tokens (good balance)
- **Llama 3.1 70B**: ~$0.35/M tokens (budget option)

### Quick Setup

In [ ]:
# Use OpenRouter for cost-effective genomics analysis
from ai_data_science_team.utils.openrouter import get_openrouter_llm

# Set your OpenRouter API key
os.environ['OPENROUTER_API_KEY'] = "sk-or-v1-..."

# Recommended: Claude 3.5 Sonnet for genomics (best quality/cost ratio)
llm_openrouter = get_openrouter_llm(
    model="anthropic/claude-3.5-sonnet",
    temperature=0
)

# Or use budget model for large-scale mutation screening:
# llm_openrouter = get_openrouter_llm("anthropic/claude-3-haiku")

llm_openrouter

In [ ]:
# Estimate costs for typical genomics analysis
from ai_data_science_team.utils.openrouter import get_cost_estimate

# Typical genomics analysis: ~50K input tokens, ~10K output tokens
estimate_claude = get_cost_estimate(
    model="anthropic/claude-3.5-sonnet",
    input_tokens=50000,
    output_tokens=10000
)

estimate_haiku = get_cost_estimate(
    model="anthropic/claude-3-haiku",
    input_tokens=50000,
    output_tokens=10000
)

print("Estimated cost for typical genomics analysis (100 samples, 500 genes):")
print(f"  Claude 3.5 Sonnet: ${estimate_claude['total']:.4f}")
print(f"  Claude 3 Haiku:    ${estimate_haiku['total']:.4f}")
print(f"  Savings with Haiku: ${estimate_claude['total'] - estimate_haiku['total']:.4f} ({((estimate_claude['total'] - estimate_haiku['total']) / estimate_claude['total'] * 100):.0f}%)")

# Use with GenomicsAnalysisAgent - exactly the same interface!
# genomics_agent = GenomicsAnalysisAgent(
#     model=llm_openrouter,  # Use OpenRouter LLM
#     gene_column="gene",
#     mutation_column="mutation_type"
# )

In [ ]:
# Option 3: Ultra-Budget Models for Massive Genomics Datasets
# When analyzing 10,000+ samples, these models save HUNDREDS of dollars

# DeepSeek Chat - Best for massive mutation screening
llm_deepseek = get_openrouter_llm("deepseek/deepseek-chat")

# Qwen 2.5 72B - Alibaba model, great for large-scale pathway analysis
llm_qwen = get_openrouter_llm("qwen/qwen-2.5-72b-instruct")

# Yi Large - Strong performance, ultra-low cost
llm_yi = get_openrouter_llm("01-ai/yi-large")

# Real-world example: TCGA pan-cancer analysis (50,000 mutations)
# - OpenAI GPT-4o cost: ~$150
# - DeepSeek Chat cost: ~$0.50
# - SAVINGS: $149.50 (99.7% cheaper!)

# Strategy: Use ultra-budget models for screening, then validate top hits with premium models
# This "funnel approach" maximizes both quality AND cost savings!

print("Ultra-budget genomics models ready!")
print("Perfect for pan-cancer mutation screening and large cohort analysis")

In [ ]:
# Option 4: Kimi K2 for Long-Form Genomic Reports
# Ideal for comprehensive variant interpretation and multi-gene panel analysis

# Kimi K2 - Moonshot AI's long-context specialist
llm_kimi = get_openrouter_llm("moonshot/kimi-k2")

# Perfect for:
# - Long-form genomic variant interpretation reports (VCF annotations)
# - Multi-gene panel analysis (50+ genes with detailed annotations)
# - Comprehensive pathway enrichment reports
# - Integrated multi-omics analysis (genomics + transcriptomics + proteomics)

# Real-world example: Comprehensive genomic report (20,000 tokens)
# - OpenAI GPT-4o: ~$0.50
# - Kimi K2: ~$0.16 (68% cheaper!)

# Combined with ultra-budget screening strategy:
# 1. Screen 50,000 mutations with DeepSeek ($0.50)
# 2. Detailed analysis of top 100 variants with Kimi K2 ($0.08)
# 3. Final validation with Claude 3.5 Sonnet ($0.30)
# Total: $0.88 vs $150+ with OpenAI only!

print("Kimi K2 ready for long-context genomics analysis!")
print("Ideal for comprehensive variant reports and multi-gene panels")

## Generate Synthetic Cancer Mutation Data

We'll create synthetic somatic mutation data mimicking a cancer genomics study with:
- Known cancer driver genes (TP53, KRAS, PIK3CA, etc.)
- Various mutation types (missense, nonsense, frameshift)
- Variant allele frequencies (VAF)
- Genomic positions

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)

# Define common cancer genes with different mutation rates
cancer_genes = {
    # High frequency drivers
    'TP53': 0.20,      # Tumor suppressor
    'KRAS': 0.15,      # Oncogene
    'PIK3CA': 0.12,    # Oncogene
    'PTEN': 0.10,      # Tumor suppressor
    'BRAF': 0.08,      # Oncogene
    # Moderate frequency
    'EGFR': 0.06,
    'BRCA1': 0.05,
    'BRCA2': 0.05,
    'APC': 0.05,
    'CDKN2A': 0.04,
    'RB1': 0.04,
    'NRAS': 0.03,
    'IDH1': 0.03,
    # Low frequency but important
    'ATM': 0.02,
    'SMAD4': 0.02,
    'VHL': 0.02,
    'NF1': 0.02,
}

# Passenger genes (background noise)
passenger_genes = [
    'TTN', 'MUC16', 'OBSCN', 'AHNAK2', 'SYNE1',  # Large genes
    'FLG', 'MUC4', 'DNAH5', 'DNAH11', 'FAT4',
    'CSMD3', 'LRP1B', 'PCLO', 'ZFHX4', 'NAV3'
]

# Mutation types and their frequencies
mutation_types = {
    'Missense': 0.65,
    'Nonsense': 0.15,
    'Frameshift': 0.10,
    'Splice_Site': 0.05,
    'Inframe_Insertion': 0.03,
    'Inframe_Deletion': 0.02,
}

# Number of samples (patients)
n_samples = 100

# Generate mutations
mutations = []

for sample_id in range(1, n_samples + 1):
    sample_name = f'SAMPLE_{sample_id:03d}'
    
    # Each sample gets 10-50 mutations
    n_mutations = np.random.randint(10, 51)
    
    for _ in range(n_mutations):
        # 70% chance of driver gene, 30% passenger
        if np.random.random() < 0.7:
            # Select driver gene weighted by frequency
            gene = np.random.choice(
                list(cancer_genes.keys()),
                p=list(cancer_genes.values())
            )
        else:
            # Select passenger gene uniformly
            gene = np.random.choice(passenger_genes)
        
        # Select mutation type
        mut_type = np.random.choice(
            list(mutation_types.keys()),
            p=list(mutation_types.values())
        )
        
        # Generate genomic position
        chromosome = np.random.choice(range(1, 23))  # Chr 1-22
        position = np.random.randint(1000000, 200000000)
        
        # Generate VAF (variant allele frequency)
        # Clonal mutations: VAF ~0.5, subclonal: lower
        vaf = np.random.beta(2, 2) * 0.6 + 0.1  # Range 0.1-0.7
        
        # Coverage depth
        depth = int(np.random.lognormal(5, 0.5))  # Mean ~150x
        
        mutations.append({
            'sample_id': sample_name,
            'gene': gene,
            'chromosome': str(chromosome),
            'position': position,
            'mutation_type': mut_type,
            'vaf': round(vaf, 3),
            'depth': depth,
        })

# Create DataFrame
mutation_data = pd.DataFrame(mutations)

print(f"Generated {len(mutation_data)} mutations across {n_samples} samples")
print(f"Unique genes mutated: {mutation_data['gene'].nunique()}")
print(f"Average mutations per sample: {len(mutation_data)/n_samples:.1f}")

mutation_data.head(15)

In [ ]:
# View summary statistics
print("Top 10 most frequently mutated genes:")
print(mutation_data['gene'].value_counts().head(10))

print("\nMutation type distribution:")
print(mutation_data['mutation_type'].value_counts())

## Example 1: Driver Gene Identification & Pathway Analysis

In [ ]:
# Initialize Genomics Analysis Agent
genomics_agent = GenomicsAnalysisAgent(
    model=llm,
    gene_column="gene",
    mutation_column="mutation_type",
    log=True,
    log_path="logs/",
    n_samples=20
)

genomics_agent

In [ ]:
# Perform comprehensive genomics analysis
genomics_agent.invoke_agent(
    data_raw=mutation_data,
    user_instructions="""
    Identify cancer driver genes from the mutation data.
    Compare to known cancer genes (TP53, KRAS, PIK3CA, etc.).
    Perform pathway enrichment analysis.
    Calculate tumor mutational burden (TMB).
    Create a bar plot showing the top 15 most frequently mutated genes.
    Flag which genes are known cancer drivers.
    """,
    max_retries=3
)

In [ ]:
# Get genomics analysis results
genomics_results = genomics_agent.get_genomics_results()
genomics_results

In [ ]:
# Display the mutation frequency plot
fig = genomics_agent.get_plotly_graph()
if fig:
    fig.show()

In [ ]:
# View the generated genomics analysis code
genomics_agent.get_genomics_analyzer_function(markdown=True)

## Example 2: Mutation Type Analysis

In [ ]:
# Create agent for mutation type analysis
mut_type_agent = GenomicsAnalysisAgent(
    model=llm,
    gene_column="gene",
    mutation_column="mutation_type",
    log=True
)

# Analyze mutation types by gene
mut_type_agent.invoke_agent(
    data_raw=mutation_data,
    user_instructions="""
    Analyze the distribution of mutation types (missense, nonsense, frameshift, etc.).
    Break down mutation types for the top 10 most frequently mutated genes.
    Create a stacked bar chart showing mutation type distribution by gene.
    Identify genes with high proportions of loss-of-function mutations (nonsense, frameshift).
    """
)

In [ ]:
# Get mutation type analysis
mut_type_results = mut_type_agent.get_genomics_results()
mut_type_results

In [ ]:
# View mutation type visualization
mut_fig = mut_type_agent.get_plotly_graph()
if mut_fig:
    mut_fig.show()

## Example 3: Pathway Enrichment Analysis

In [ ]:
# Create agent for pathway analysis
pathway_agent = GenomicsAnalysisAgent(
    model=llm,
    gene_column="gene",
    mutation_column="mutation_type"
)

# Perform pathway enrichment
pathway_agent.invoke_agent(
    data_raw=mutation_data,
    user_instructions="""
    Perform cancer pathway enrichment analysis.
    Group mutations by biological pathway:
    - TP53 pathway
    - RAS/RAF/MEK/ERK signaling
    - PI3K/AKT/mTOR pathway
    - DNA repair pathways
    - Cell cycle regulation
    
    Calculate the number of samples with mutations in each pathway.
    Create a heatmap or bar plot showing pathway mutation frequencies.
    """
)

In [ ]:
# Get pathway enrichment results
pathway_results = pathway_agent.get_genomics_results()
pathway_results

In [ ]:
# Visualize pathway enrichment
pathway_fig = pathway_agent.get_plotly_graph()
if pathway_fig:
    pathway_fig.show()

## Example 4: Tumor Mutational Burden (TMB) Analysis

In [ ]:
# Create agent for TMB calculation
tmb_agent = GenomicsAnalysisAgent(
    model=llm,
    gene_column="gene",
    mutation_column="mutation_type"
)

# Calculate TMB per sample
tmb_agent.invoke_agent(
    data_raw=mutation_data,
    user_instructions="""
    Calculate tumor mutational burden (TMB) for each sample.
    TMB = number of somatic mutations per sample.
    
    Provide:
    - TMB distribution across all samples
    - Identify high TMB samples (>30 mutations)
    - Identify low TMB samples (<15 mutations)
    - Create a histogram of TMB values
    
    Note: In real exome sequencing, divide by ~30 Mb for mutations/Mb.
    """
)

In [ ]:
# Get TMB results
tmb_results = tmb_agent.get_genomics_results()
tmb_results

In [ ]:
# Visualize TMB distribution
tmb_fig = tmb_agent.get_plotly_graph()
if tmb_fig:
    tmb_fig.show()

## Example 5: Actionable Mutation Discovery

In [ ]:
# Create agent for actionable mutations
actionable_agent = GenomicsAnalysisAgent(
    model=llm,
    gene_column="gene",
    mutation_column="mutation_type"
)

# Identify targetable mutations
actionable_agent.invoke_agent(
    data_raw=mutation_data,
    user_instructions="""
    Identify actionable/targetable mutations suitable for precision medicine.
    
    Focus on genes with FDA-approved targeted therapies:
    - BRAF (V600E) → BRAF inhibitors
    - EGFR → EGFR inhibitors
    - PIK3CA → PI3K inhibitors
    - KRAS (G12C) → KRAS inhibitors
    - BRCA1/2 → PARP inhibitors
    
    For each sample, report:
    - Which actionable genes are mutated
    - Number of potentially targetable alterations
    - Create a summary table of samples with actionable mutations
    """
)

In [ ]:
# Get actionable mutations
actionable_results = actionable_agent.get_genomics_results()
actionable_results

## Example 6: Sample-Level Analysis (Oncoprint)

In [ ]:
# Create agent for oncoprint-style visualization
onco_agent = GenomicsAnalysisAgent(
    model=llm,
    gene_column="gene",
    mutation_column="mutation_type"
)

# Create mutation matrix
onco_agent.invoke_agent(
    data_raw=mutation_data,
    user_instructions="""
    Create an oncoprint-style mutation matrix.
    
    For the top 15 most frequently mutated genes:
    - Show which samples have mutations in each gene
    - Color-code by mutation type
    - Calculate mutation frequency for each gene
    - Show co-occurrence patterns
    
    Create a heatmap showing mutation presence/absence across samples.
    """
)

In [ ]:
# Get oncoprint data
onco_results = onco_agent.get_genomics_results()
onco_results.head(20)

In [ ]:
# View oncoprint visualization
onco_fig = onco_agent.get_plotly_graph()
if onco_fig:
    onco_fig.show()

## Example 7: VAF-based Clonality Analysis

In [ ]:
# Create agent for clonality analysis
vaf_agent = GenomicsAnalysisAgent(
    model=llm,
    gene_column="gene",
    mutation_column="mutation_type"
)

# Analyze variant allele frequencies
vaf_agent.invoke_agent(
    data_raw=mutation_data,
    user_instructions="""
    Analyze variant allele frequencies (VAF) to infer clonality.
    
    Categorize mutations by VAF:
    - Clonal: VAF > 0.35 (present in most tumor cells)
    - Subclonal: VAF 0.15-0.35 (present in subset of cells)
    - Low frequency: VAF < 0.15 (rare variants)
    
    For each category:
    - Which genes are most frequently clonal vs subclonal?
    - Create VAF distribution histogram
    - Identify samples with high subclonal diversity
    """
)

In [ ]:
# Get VAF analysis
vaf_results = vaf_agent.get_genomics_results()
vaf_results

In [ ]:
# View VAF distribution
vaf_fig = vaf_agent.get_plotly_graph()
if vaf_fig:
    vaf_fig.show()

## Summary

The **GenomicsAnalysisAgent** provides:

✅ **Automated mutation analysis** - No manual coding required  
✅ **Driver gene identification** - Compare to known cancer genes  
✅ **Pathway enrichment** - Biological pathway-level insights  
✅ **TMB calculation** - For immunotherapy eligibility  
✅ **Actionable mutations** - Precision medicine targets  
✅ **Interactive visualizations** - Plotly charts for presentations  
✅ **Reproducible** - All code is logged and can be reused  

## Integration with Survival Analysis

Combine genomics with survival analysis:

```python
from ai_data_science_team.ml_agents import SurvivalAnalysisAgent, GenomicsAnalysisAgent

# 1. Identify driver mutations
genomics_agent.invoke_agent(
    data_raw=mutation_data,
    user_instructions="Identify TP53 mutations"
)

# 2. Add mutation status to clinical data
clinical_data['TP53_mutated'] = ...  # Based on genomics results

# 3. Analyze survival by mutation status
survival_agent.invoke_agent(
    data_raw=clinical_data,
    user_instructions="Compare survival: TP53 mutated vs wild-type"
)
```

## Next Steps

1. **Real cancer datasets**: Use with TCGA, ICGC, or your own sequencing data
2. **VCF file integration**: Load variant call format files
3. **Multi-omics**: Combine with gene expression, copy number data
4. **Database integration**: Connect to COSMIC, ClinVar, OncoKB
5. **Custom pathways**: Define institution-specific gene sets